# Lesson 43 · How the STFT works — give sound a timeline: the short-time Fourier transform and the eternal time–frequency trade-off

**Goal**: get a firm grip on the core intuition of the STFT first: how it cuts 1-D sound into frame after frame and turns that into a 2-D time–frequency matrix. Along the way, understand the trade-off between window length, hop and time/frequency resolution, and implement `frame_signal()`.

> **The three parameters in plain words** (following on from the bin/hop of L03):
> - `n_fft` / window length: how long each slice is — longer means finer frequency and blurrier time
> - `hop`: how far the next slice jumps — smaller means a denser time axis
> - overlap: set by window length and hop together; not a third independent knob
> One comparison plot of the same signal at two window lengths beats memorizing formulas.

🔗 **Aurora connection**: `aurora.audio.stft.frame_signal()` and `aurora.audio.stft.stft()` are the target implementations this lesson lines up with; the code lives in `src/aurora/audio/stft.py`.

← **Previous lesson** · [L42 · The FFT drawn](L42_visual_fft.ipynb)

> Last lesson covered **the FFT drawn**: the butterfly diagram, plus the spectra of a pure tone, a chord and noise side by side.
> This lesson explores **how the STFT works**.

## This lesson's story: the FFT's blind spot — *when* does each frequency show up?

You're listening to a piano piece: it opens in C major, modulates to G in the middle and comes back to C at the end. Take one FFT of the whole piece and you'll see that the frequencies of both C and G are there — but which came first? You can't tell at all.

That's the FFT's blind spot: **it knows "which frequencies", but not "when"**.

The STFT's (Short-Time Fourier Transform's) fix is as direct as it gets — like flipping through slides, cut the signal into short frames and FFT each one on its own:

```
STFT[m, k] = FFT( x[m·hop : m·hop + win_len] × window )[k]
```

- `m`: frame m (the time coordinate)
- `k`: frequency bin k (the frequency coordinate)
- `hop`: the step between two adjacent frames (sets the time resolution)

The result is a 2-D matrix: rows are time frames, columns are frequency bins. Turn it into a heatmap and you have a **spectrogram** — the standard format for visualizing audio.

**The time–frequency trade-off (the uncertainty principle in practice)**:

| win_len (frame length) | Δf = sr/win_len | frame duration = win_len/sr | Good for |
|---|---|---|---|
| 128 samples | 62.5 Hz/bin | 16 ms/frame | precise timing (transients) |
| 512 samples | 15.6 Hz/bin | 64 ms/frame | precise frequency (pitch) |
| 2048 samples | 3.9 Hz/bin | 256 ms/frame | very precise frequency (singing) |

The longer the frame, the finer the frequency; the shorter the frame, the sharper the timing — you can't have both.

Your task in this lesson: implement `frame_signal(x, win_len, hop)` — cut a 1-D signal into a 2-D matrix of shape `(n_frames, win_len)`.

## 🤔 Why did engineers invent this?
- **What happens without it?** Take **one** FFT of a whole recording and you only learn "which frequencies occurred", not "at which second" — a bird's chirp, the burst of a consonant, a key change in a piece of music all get flattened into one average spectrum.
- **What real problem does it solve?** Cut the long signal into short windows, frame after frame, and FFT each one, and every frequency gets a timestamp. That's the point of the "sliding window": **one FFT can't give you a time axis; only a series of sliding FFTs can draw a time–frequency plot**.
- **Where does it come back later?** Assembling `stft()` yourself in L44, the spectrogram in L45, the whole mel / MFCC pipeline in L46–L50 — they all stand on this frame-by-frame time–frequency matrix.

## Parts you already have (no need to reinvent them)

| Part | From lesson | Aurora function |
|---|---|---|
| FFT | L39 | `aurora.audio.transforms.fft` |
| window function | L36 | `aurora.audio.windows.hann` / `get_window` |
| framing | **this lesson's TODO** | `aurora.audio.stft.frame_signal` |

L44's job is **assembly**: `frame_signal` → `× window` → `rfft/fft`,
not re-deriving the FFT or redesigning the window.


In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from aurora.audio.io import sine

## Concept 1: the definition of the STFT

```
STFT[m, k] = FFT( x[m*hop : m*hop+win_len] * window )[k]
```

- `m`: frame index (time axis)
- `k`: frequency-bin index (frequency axis)
- `window`: the weighting function (Hann, etc.) that suppresses spectral leakage from the frame edges

The input is a 1-D signal of length `N`; the output is a complex matrix of shape `(n_frames, win_len//2+1)`. Take the magnitude for the magnitude spectrum and the angle for the phase spectrum.

### Why multiply by a window? — where spectral leakage comes from and how to suppress it

**What is leakage?** Picture a clean 1.1 kHz sine wave sampled at 8 kHz, and collect 32 samples (4 milliseconds in all). Ideally the FFT would show a single peak at that frequency — but in reality you'll see lots of small "sidelobes" around the peak: energy that shouldn't be there, scattered across neighboring frequency bins. That's **spectral leakage**.

**Where does leakage come from?** Mathematically, the FFT of an infinitely long pure tone $\sin(2\pi f t)$ is nonzero in just one bin. But you only sampled 32 points — which is the same as **chopping the signal off abruptly at both ends**. It's like suddenly stopping mid-performance with a "click": that click is itself a high-frequency transient. The FFT treats the discontinuity as spurious frequency content and spreads it across the surrounding bins.

**How does a window suppress it?** The Hann window (and others like Hamming and Blackman) are bell-shaped curves: close to 1 in the middle, tapering to 0 at the ends. Multiplying the signal by one of these is like **a gentle fade-out instead of a hard cut** — the ends taper off naturally with no sharp edge, so the FFT doesn't pick up false high frequencies.

That's why the standard STFT formula always multiplies by a window:
$$\text{STFT}[m, k] = \text{FFT}( x[m \cdot \text{hop} : m \cdot \text{hop} + \text{win\_len}] \times \text{window} )[k]$$

The price is that the window widens the peak a little (slightly lower resolution), but the sidelobes are pushed down to much lower energy and the true spectrum comes out clearer.

In [ ]:
# Demo: FFT a single frame by hand and see how it relates to np.fft.rfft
x_demo = sine(freq=440, duration=0.01, sample_rate=8000)  # 80 samples
win_len = 32
window = np.hanning(win_len)

frame = x_demo[:win_len] * window
spectrum = np.fft.rfft(frame)  # reference value; we'll hand-write the FFT later

print(f'Frame length: {win_len}, frequency bins: {len(spectrum)}')
print(f'First 5 bins of the magnitude spectrum: {np.abs(spectrum[:5]).round(4)}')

## Concept 2: the time–frequency trade-off (the uncertainty principle on discrete signals)

### The two resolutions (keep them clearly apart)

```
frequency resolution = sr / win_len   (Hz/bin)
time resolution      = hop / sr        (seconds/frame)
```

What's easy to mix up here is that **"time" means two things**:
- **The duration of each frame** (frame duration) = `win_len / sr` (seconds/frame) — the time span the samples of one frame cover. For example, with sr=8000 and win_len=512, each frame captures 512/8000 = 64 milliseconds of audio.
- **The time between adjacent frames** (frame hop time) = `hop / sr` (seconds/frame) — the time step from one frame to the next. If hop=256, that's 256/8000 = 32 milliseconds per jump.

**Time resolution here means the spacing between frames (hop/sr), not the length of each frame.** Intuitively: the smaller the step between frames, the more finely you can track how the signal changes over time (high time resolution); the bigger the step, the more you only see "big" changes over time (low time resolution).

### Why these formulas?

**Why is the frequency resolution sr/win_len?**
- the FFT has win_len outputs (or win_len//2+1 that matter for a real signal)
- those values cover the frequency range from 0 to sr/2
- so each frequency bin is (sr/2) / (win_len//2) ≈ sr/win_len Hz wide
- the longer the window, the more bins, the narrower each bin, the higher the frequency resolution

**Why is the time resolution hop/sr?**
- adjacent frames are hop samples apart
- samples are 1/sr seconds apart
- so adjacent frames are hop/sr seconds apart
- the smaller the hop, the denser the frames, the higher the time resolution

### The time–frequency trade-off table

| win_len (frame length) | Δf = sr/win_len | Δt = hop/sr | hop setting | Good for |
|---|---|---|---|---|
| 128 samples | 62.5 Hz/bin | 4 ms | win_len//4 | precise timing (transients) |
| 512 samples | 15.6 Hz/bin | 16 ms | win_len//4 | a general-purpose balance |
| 2048 samples | 3.9 Hz/bin | 64 ms | win_len//4 | precise frequency (pitch) |

Longer frame → finer frequency; shorter frame → sharper timing. You can't optimize both at once.

- bigger `win_len` → more samples per frame → higher frequency resolution, but each frame spans more time (fuzzier time localization)
- smaller `hop` → shorter steps between frames → higher time resolution, but more computation (more frames to compute)
- you can't have both at once — that's the uncertainty principle showing up on discrete signals

In [ ]:
# Demo: frequency/time resolution at different win_len
sr = 8000
for win_len in [128, 512, 2048]:
    freq_res = sr / win_len
    for hop in [win_len//4, win_len//2]:
        time_res_ms = hop / sr * 1000
        print(f'win={win_len:5d}  hop={hop:5d}  '
              f'freq resolution={freq_res:.1f}Hz  time resolution={time_res_ms:.1f}ms')

## Seeing the time–frequency trade-off: one signal, three window lengths

Below, **one synthetic signal** (a steady pure tone + a momentary "click" pulse) is drawn as three spectrograms with `win_len=256 / 512 / 2048`. The three plots show the same trade-off:

- **Short window (256)**: the vertical bar is crisp — **time** is accurate (you can pin down the "click"), but the pure tone's line is thick in frequency.
- **Long window (2048)**: the horizontal line is sharp — **frequency** is accurate (the tone's line is thin), but the "click" gets smeared out and the timing goes blurry.
- **Medium window (512)**: a compromise. Real systems pick a balance point somewhere along this axis.

In one line: **a long window buys frequency precision, a short window buys time precision — you can't have your cake and eat it too**.

## Concept 2.5: deriving the frame-count formula (why 1 + (N - win_len) // hop?)

**Intuition**: cut the signal into overlapping frames starting at positions 0, hop, 2·hop, 3·hop, ..., each win_len long. The key point is that **the last frame has to be complete** (exactly win_len long, not short).

**The derivation**:

Say frame m starts at `m * hop` (m=0, 1, 2, ...); then frame m covers samples `[m*hop, m*hop + win_len)`.

For this frame to be complete we need:
$$m \cdot \text{hop} + \text{win\_len} \leq N$$

where N is the total number of samples. Solve for m:
$$m \leq \frac{N - \text{win\_len}}{\text{hop}}$$

The largest m is:
$$m_{\max} = \left\lfloor \frac{N - \text{win\_len}}{\text{hop}} \right\rfloor$$

Since frames are numbered from 0, the total number of frames is $m_{\max} + 1$:
$$n_{\text{frames}} = 1 + \left\lfloor \frac{N - \text{win\_len}}{\text{hop}} \right\rfloor = 1 + \frac{N - \text{win\_len}} {\text{hop}}$$
(with the division as integer division, written `//`)

**Checking by hand**:
- N=10, win=4, hop=2: $m_{\max} = \lfloor (10-4)/2 \rfloor = 3$, so 4 frames, starting at 0,2,4,6 ✓
- N=100, win=32, hop=16: $m_{\max} = \lfloor (100-32)/16 \rfloor = 4$, so 5 frames ✓

This formula counts only **the frames you can take in full**. Leftover data shorter than win_len at the end is discarded (center=False mode); if you need to cover the whole signal, you can zero-pad the end to $(n_{\text{frames}} - 1) \cdot \text{hop} + \text{win\_len}$ samples.

In [ ]:
# Static plot: the time–frequency trade-off for one signal at win=256/512/2048 (self-contained; doesn't depend on the TODO above)
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sr, dur = 8000, 1.0
t = np.linspace(0, dur, int(sr * dur), endpoint=False)
sig = np.sin(2 * np.pi * 1000 * t)          # steady pure tone → should be a thin line on the frequency axis
click = int(0.5 * sr)
sig[click:click + 8] += 6.0                 # a momentary pulse at 0.5 s → should be a vertical bar on the time axis

def quick_spec(x, win, hop):
    w = np.hanning(win)
    n_frames = 1 + (len(x) - win) // hop
    cols = [np.abs(np.fft.rfft(x[m * hop:m * hop + win] * w))
            for m in range(n_frames)]
    return 20 * np.log10(np.array(cols).T + 1e-6)   # (freq, time) dB

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, win in zip(axes, (256, 512, 2048)):
    S = quick_spec(sig, win, win // 4)
    ax.imshow(S, origin="lower", aspect="auto", cmap="magma",
              extent=[0, dur, 0, sr / 2])
    ax.set_title(f"win_len={win}\nΔf={sr/win:.0f}Hz  frame={win/sr*1000:.0f}ms")
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("freq (Hz)")
plt.tight_layout()
plt.show()
print("window↑: horizontal line (frequency) sharper, vertical bar (time) blurrier; window↓: the reverse — that's the time–frequency trade-off.")

### Details: the transpose and the log scale (why does a spectrogram look like this?)

In the code above:
```python
cols = [np.abs(np.fft.rfft(...)) for m in range(n_frames)]  # FFT frame by frame
S = 20 * np.log10(np.array(cols).T + 1e-6)  # transpose, take the log
```

There are two "whys" here:

**Why transpose with `.T`?**
- the frame-by-frame FFT gives `cols` with shape `(n_frames, n_freqs)`: each row is the spectrum of one time frame
- when an image is drawn, the horizontal axis is columns and the vertical axis is rows
- we want **time** (frames) on the horizontal axis and **frequency** on the vertical axis
- so we transpose to `(n_freqs, n_frames)`, making frequencies the rows and frames the columns

**Why take the log `20 * np.log10(...)`?**
- this is the **decibel (dB) scale**: $\text{dB} = 20 \log_{10}(\text{amplitude})$
- human loudness perception is logarithmic, not linear. Make a sound's amplitude 10 times larger and it sounds only somewhat louder — nowhere near "10 times as loud"
- in decibels, faint sounds become visible (they're boosted on the display), and loud ones don't "blow out" the whole color scale
- the `1e-6` avoids log(0), so nothing blows up numerically
- when reading a spectrogram: a bin with magnitude 1 is 0 dB; magnitude 0.1 is -20 dB; that "faint" -20 dB signal is still easy to see on a dB plot

**Converting a frequency bin to Hz**:
- in the STFT output, frequency bin k corresponds to:
$$f_k = k \times \frac{\text{sr}}{\text{win\_len}} \quad (\text{Hz})$$
- for example, with sr=8000 and win_len=512, Δf = 15.625 Hz/bin, so bin k=10 is at 10 × 15.625 = 156.25 Hz

## Concept 3: handling frame boundaries and padding (what the center parameter means)

**The basic formula** (center=False mode):
```
n_frames = 1 + (len(x) - win_len) // hop
```

This counts only **complete frames**. A last frame shorter than `win_len` is simply dropped.

**Two ways to handle the boundaries**:

| Mode | What it does | Frame count | When to use |
|------|------|------|--------|
| **center=False** | take only complete frames; drop a short tail | fewer (by the formula) | real-time processing, no need to cover everything |
| **center=True** | pad the start and end (often by reflection rather than zeros) so the signal is centered | a few more | when you need to cover the whole signal or make reconstruction easier |

**An example** (N=10, win_len=4, hop=2):
- **center=False**: only 4 frames (starting at 0,2,4,6); the frame starting at 6 covers samples 6-9, which is fine ✓; a frame starting at 8 would need samples 8-11, but the signal stops at 9, so it's dropped
- **center=True**: after padding both ends N is larger, and you may get 5 frames or more

**Why pad?** In signal processing you sometimes need to make sure the STFT result can fully reconstruct the original signal (this involves the COLA condition; see the section at the end of this lesson). Padding guarantees no information is lost. But if you're only analyzing and don't need to reconstruct, center=False is enough.

In [ ]:
# Demo: the frame-count formula
for n, w, h in [(10,4,2),(100,32,16),(22050,2048,512)]:
    n_frames = 1 + (n - w) // h
    print(f'len={n:6d}  win={w:5d}  hop={h:5d}  → n_frames={n_frames}')

## 1. ✏️ Implement `frame_signal(x, win_len, hop)`

**Core formula**: frame m = `x[m*hop : m*hop + win_len]`

**A three-step route**:

| Step | Code idea | Notes |
|---|---|---|
| 1 | `n_frames = 1 + (len(x) - win_len) // hop` | frame-count formula (number of complete frames) |
| 2 | `frames = np.zeros((n_frames, win_len))` | preallocate the output matrix |
| 3 | `frames[m] = x[m*hop : m*hop+win_len]` | slice frame by frame (loop m=0..n_frames-1) |

**Acceptance criteria**:
- `frame_signal(np.arange(10), win_len=4, hop=2)` → shape `(4, 4)`
- `frames[0] = [0,1,2,3]`, `frames[1] = [2,3,4,5]` (stepping by hop=2)
- every frame has the same length (a leftover tail shorter than win_len is not taken)

In [ ]:
def frame_signal(x: np.ndarray, win_len: int, hop: int) -> np.ndarray:
    """Cut a 1-D signal into overlapping frames; return a matrix of shape (n_frames, win_len).

    Args:
        x: input signal, shape (N,)
        win_len: length of each frame (in samples)
        hop: step between adjacent frames (in samples)

    Returns:
        frames: real matrix of shape (n_frames, win_len), one frame per row

    Reference implementations (pick one):

    **Option A: a simple loop (easy to follow; recommended for beginners)**
    ```
    n_frames = 1 + (len(x) - win_len) // hop
    frames = np.zeros((n_frames, win_len))
    for m in range(n_frames):
        frames[m] = x[m * hop : m * hop + win_len]
    return frames
    ```

    **Option B: stacked slices (no explicit loop)**
    ```
    n_frames = 1 + (len(x) - win_len) // hop
    # use np.lib.stride_tricks.as_strided, an advanced trick
    # or numpy fancy indexing: np.array([x[m*hop:m*hop+win_len] for m in range(n_frames)])
    ```
    """
    # ✏️ TODO: implement option A or B above
    raise NotImplementedError("implement frame_signal")

In [ ]:
# Check frame_signal
try:
    result = frame_signal(np.arange(10), 4, 2)
except (NotImplementedError, TypeError):
    result = None
if result is None or result is ...:
    print('⬜ frame_signal not implemented yet')
else:
    assert result.shape == (4, 4), f'wrong shape: {result.shape}'
    assert list(result[0]) == [0, 1, 2, 3], f'frame 0 is wrong: {result[0]}'
    assert list(result[1]) == [2, 3, 4, 5], f'frame 1 is wrong: {result[1]}'
    assert list(result[3]) == [6, 7, 8, 9], f'frame 3 is wrong: {result[3]}'
    print('✅ frame_signal check passed: shape', result.shape)
    print('Frame matrix:\n', result)

## Getting a feel for the numbers · 1 second @ 16 kHz (compare with the measurement in the next cell)

| win_len | hop | n_frames (center=False) | time resolution (ms) | frequency resolution (Hz) |
|---|---:|---:|---:|---:|
| 512 | 128 | 122 | 8.0 | 31.25 |
| 2048 | 512 | 28 | 32.0 | 7.81 |

Note: with `center=True` the frame count is slightly higher (the ends are padded); trust what `aurora.stft.frame_signal` actually returns.


In [ ]:
import numpy as np
from aurora.audio.stft import frame_signal as _af_frame

sr = 16_000
x = np.zeros(sr, dtype=float)

rows = []
for win_len, hop in ((512, 128), (2048, 512)):
    frames = _af_frame(x, win_len, hop, center=False)
    n = frames.shape[0]
    formula = 1 + (len(x) - win_len) // hop
    t_ms = hop / sr * 1000
    f_hz = sr / win_len
    rows.append((win_len, hop, n, formula, t_ms, f_hz))
    assert n == formula, f"win={win_len} hop={hop}: got {n}, expected {formula}"

for win_len, hop, n, formula, t_ms, f_hz in rows:
    print(
        f"win={win_len:4d} hop={hop:3d}  n_frames={n:3d}  "
        f"Δt={t_ms:4.1f}ms  Δf={f_hz:5.2f}Hz"
    )
print("✅ n_frames in the table matches frame_signal(center=False)")

## Parameter experiment: how win_len affects time resolution

Take the same 0.5-second signal (a 440 Hz sine; sr=8000, N=4000 samples) and set `win_len=128` (16 ms) and `win_len=2048` (256 ms), with `hop` set to `win_len//4` in both cases.

**What you should see**:
- `win_len=128` → many frames (about **122 frames**; formula: 1+(4000-128)//32=122), high time resolution, but few frequency bins per frame (**65**, i.e. win_len//2+1)
- `win_len=2048` → few frames (about **4 frames**; formula: 1+(4000-2048)//512=4), many frequency bins (**1025**, i.e. win_len//2+1), but coarse time granularity
- the ratio of frame counts (small signal, N=4000) ≈ 122/4 ≈ **30.5×**; for a long signal (N≫win_len) it approaches the hop ratio 512/32 = **16×**

> ⚠️ Note: N=4000 is just a small demo signal. A 2048 window fits only 4 complete frames into 4000 samples, which pushes the ratio far above the asymptotic 16. The asymptotic pattern only becomes clear on a long signal (e.g. sr=8000 × 10 seconds = 80000 samples).

In [ ]:
# Guard: if frame_signal isn't implemented yet, skip this cell instead of crashing
try:
    _guard = frame_signal(np.arange(10), 4, 2)
except (NotImplementedError, TypeError):
    _guard = None
if _guard is None or _guard is ...:
    print('⬜ Finish the frame_signal exercise first, then run this cell')
    raise SystemExit(0)

# Parameter experiment: how win_len affects the frame count (time resolution)
sr = 8000
duration = 0.5
x = sine(freq=440, duration=duration, sample_rate=sr)

for win_len in [128, 2048]:
    hop = win_len // 4
    frames = frame_signal(x, win_len, hop)
    freq_bins = win_len // 2 + 1
    time_res_ms = hop / sr * 1000
    freq_res_hz = sr / win_len
    print(f'win_len={win_len:5d}  hop={hop:5d}  '
          f'frames={frames.shape[0]:4d}  '
          f'freq bins={freq_bins:5d}  '
          f'time resolution={time_res_ms:.1f}ms  '
          f'freq resolution={freq_res_hz:.1f}Hz')

print()
f128 = frame_signal(x, 128, 32)
f2048 = frame_signal(x, 2048, 512)
print(f'Frame-count ratio (128 vs 2048): {f128.shape[0]} / {f2048.shape[0]} = {f128.shape[0]/f2048.shape[0]:.1f}x')

### Why does the frame-count ratio "approach" the hop ratio for long signals?

The code above shows that for N=4000 (a short signal), the frame-count ratio of `win_len=128 vs 2048` is 122/4 ≈ 30.5, far above the hop ratio — 32/512 = 1/16 ≈ 6.25% (or, put the other way round, 512/32 = 16×). But as the signal gets longer, the ratio gradually approaches the hop ratio. Why?

**The math**:
- for a signal of length N, $n_1 = 1 + (N - 128) / 32$ and $n_2 = 1 + (N - 2048) / 512$
- when N is large, the "1" hardly matters, so:
  - $n_1 \approx N / 32$
  - $n_2 \approx N / 512$
  - ratio $n_1 / n_2 \approx (N/32) / (N/512) = 512/32 = 16$ ✓ (exactly the hop ratio 512/32)

**Why is the ratio so large at N=4000 (30.5 vs 16)?**
- with `win_len=128`: `n_1 = 1 + (4000-128)/32 = 1 + 121 = 122`
- with `win_len=2048`: `n_2 = 1 + (4000-2048)/512 = 1 + 3 = 4` (only 4 frames!)
- because a 2048 window is too big for a 4000-sample signal and barely gets used, while the 128 window can still sample densely. Here the leading "1" has a big effect on the small n_2
- try again with N=80000 (sr=8000 × 10 seconds) and the ratio comes close to 16

This is a classic **edge effect**: on a short signal, the "+1" in the formula and the difference between the two windows' `win_len` both get amplified; on a long signal those deviations become negligible and the hop alone dominates.

## Wrap-up

`frame_signal(x, win_len, hop)` cuts 1-D sound into a 2-D matrix of shape `(n_frames, win_len)`; it's the first step of the STFT pipeline. `win_len` and `hop` together set the time–frequency trade-off: a big window buys frequency precision, a small hop buys time precision, and you can't optimize both at once. This function plugs straight into `aurora.audio.stft.stft()`, which multiplies every frame by a Hann window, takes its FFT, and finally stacks the results into the full complex time–frequency matrix. The next lesson (L44) implements the complete `stft()` function and draws the spectrogram of real audio.

## ✏️ Whiteboard challenge: the STFT by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: `x` has length 100, `win_len=32`, `hop=16`. How many frames are there?
(Formula: `n_frames = 1 + (100-32)//16 = ?`)

**Question 2**: What matrix should `frame_signal(np.arange(10), 4, 2)` return?
(4 rows, 4 columns; write out the values in each row)

**Question 3**: `sr=8000`, `win_len=512`:
- frequency resolution Δf = sr/win_len = ? Hz/bin
- frame duration Δt = win_len/sr = ? ms/frame

**Question 4**: Why do STFT frames usually overlap by 50% (`hop = win_len/2`) rather than not at all?
(Hint: it has to do with the Hann-window COLA condition from L41)

**Question 5**: What do the horizontal axis, the vertical axis and the color of a spectrogram each represent?

When you've worked it out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: frame-count formula
n, w, h = 100, 32, 16
n_frames_calc = 1 + (n - w) // h
assert n_frames_calc == 5
print(f"Q1 ✅  n_frames = 1 + (100-32)//16 = {n_frames_calc}")

# Q2: frame_signal(arange(10), 4, 2) by hand
x10 = np.arange(10, dtype=float)
expected = np.array([
    [0, 1, 2, 3],
    [2, 3, 4, 5],
    [4, 5, 6, 7],
    [6, 7, 8, 9],
], dtype=float)
try:
    result = frame_signal(x10, 4, 2)
    assert result is not None and result.shape == (4, 4)
    assert np.allclose(result, expected, atol=1e-12)
    print(f"Q2 ✅  frame_signal(arange(10),4,2) shape={result.shape}, frames[0]={result[0]}, frames[1]={result[1]}")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  by hand: frames[0]={expected[0]}, frames[1]={expected[1]} (frame_signal not implemented yet)")

# Q3: time/frequency resolution at sr=8000, win_len=512
sr, win_len = 8000, 512
delta_f = sr / win_len
delta_t_ms = win_len / sr * 1000
assert delta_f == 15.625
assert abs(delta_t_ms - 64.0) < 0.001
print(f"Q3 ✅  Δf={delta_f} Hz/bin, frame duration Δt={delta_t_ms:.1f} ms/frame")

# Q4: COLA check at 50% overlap (N=8 Hann window)
N4 = 8
w4 = 0.5 * (1 - np.cos(2 * np.pi * np.arange(N4) / N4))
hop4 = N4 // 2
ola = w4[:hop4] + w4[hop4:]
assert np.std(ola) < 0.05 * np.mean(ola)
print(f"Q4 ✅  Hann window at 50% overlap satisfies COLA: OLA mean={np.mean(ola):.4f}, std={np.std(ola):.6f}")

# Q5: what the spectrogram axes mean (checking the concept with an assertion)
# horizontal axis = time (frame m), vertical axis = frequency (bin k), color = |STFT[m,k]| or dB
assert True  # a concept question; here an fft check of the spectrum shape stands in
sr5, T5 = 8000, 0.032
t5 = np.linspace(0, T5, int(sr5 * T5), endpoint=False)
x5 = np.sin(2 * np.pi * 440 * t5)
X5 = np.fft.rfft(x5 * np.hanning(len(x5)))
assert np.argmax(np.abs(X5)) > 0  # the 440 Hz peak is not at DC
print(f"Q5 ✅  spectrogram: horizontal axis = time frame, vertical axis = frequency bin, color = |STFT[m,k]| (usually in dB)")
print("\n🎉 STFT whiteboard challenge passed! n_frames=1+(N-win)//hop, Δf=sr/win, frame duration=win/sr.")

In [ ]:
# ✏️ Self-review for this lesson
l43_review = {
    "stft_formula":            None,  # remember STFT[m,k]=FFT(x[m·hop:m·hop+win]×window)[k]? True/False
    "frame_signal_impl":       None,  # frame_signal implemented and passing the shape assertion? True/False
    "time_freq_tradeoff":      None,  # understand the trade-off: big window → high frequency resolution / low time resolution? True/False
    "n_frames_formula":        None,  # remember n_frames=1+(N-win_len)//hop? True/False
    "whiteboard_passed":       None,  # whiteboard challenge worked on paper? True/False
    "parts_inventory_read":    None,  # read the parts table and know that L44 is assembly? True/False
}

unfilled = [k for k, v in l43_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l43_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L43 fully cleared! On to L44: assembling frame→window→fft')

## Why 50% overlap? — the COLA condition and perfect reconstruction

**Where the problem starts**: you cut a long signal into overlapping frames, window them and FFT them, and in the end you want to **add the frames' results back together to reconstruct the original signal**. But if the window is chosen badly, adding up the windowed frames can change the amplitude — some moments get boosted, others weakened. That's bad news.

**The COLA (Constant Overlap-Add) condition**: let the window be $w(n)$. If, at the overlap you've chosen, the overlapping windows always add up to a constant, the window satisfies the COLA condition. Mathematically:
$$\sum_{m=-\infty}^{\infty} w(n + m \cdot \text{hop}) = \text{const} \quad \text{for all } n$$

This means that the signal at every moment $n$ is windowed once for each frame that covers it, and those windows always add up to the same total — so the original signal can be **reconstructed perfectly**, without distortion.

**Hann window + 50% overlap satisfies COLA**:
- a mathematical property of the (periodic) Hann window: $w(n) + w(n + N/2) = 1$ (N is the window length)
- 50% overlap means $\text{hop} = N/2$
- so $w(n) + w(n + \text{hop}) = 1$ (for all n)
- and therefore all the frames' windows add up to 1: COLA is satisfied ✓

**What it means in practice**: if you run an STFT with a Hann window and 50% overlap, process each windowed frame (say, noise suppression), and then **inverse-FFT + overlap-add**, you can reconstruct the signal without losing energy or introducing artifacts. This is essential in speech enhancement and audio effects processing.

**Other windows with other overlaps** can satisfy COLA too, but Hann + 50% is the most common combination.

---

→ **Next lesson** · [L44 · Writing the STFT](L44_stft_implement.ipynb)

> Next lesson covers **writing the STFT**: framing + windowing + FFT, checked against aurora.audio.stft.